<a href="https://colab.research.google.com/github/Atherizz/PCVK_2026/blob/main/week_9/week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# MODUL 9 - OPERASI MORFOLOGI CITRA DIGITAL
**Mata Kuliah:** Pengolahan Citra dan Visi Komputer (PCVK)  
**Jurusan:** Teknologi Informasi - D4 Teknik Informatika  
**Topik Praktikum:** Erosi, Dilasi, Opening, Closing, Top-Hat, Black-Hat (Bottom-Hat), Skeletonization, Thickening, dan Thinning

## Cell Identitas Mahasiswa

In [ ]:
NAMA = "Savero Athallah Hardiana Putra"
NIM  = "244107020116"
KELAS = "TI-3B"
MATA_KULIAH = "Pengolahan Citra dan Visi Komputer"

print("=" * 55)
print(f"Nama        : {NAMA}")
print(f"NIM         : {NIM}")
print(f"Kelas       : {KELAS}")
print(f"Mata Kuliah : {MATA_KULIAH}")
print("=" * 55)

## Persiapan: Setup Environment, Mount Drive, & Import Library
Notebook ini dirancang adaptif: dapat dijalankan langsung di **Google Colab** (dengan mounting Google Drive) maupun di lingkungan lokal (`.venv`).

In [ ]:
import os
import sys
import cv2
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import skimage
from skimage import io, filters, morphology, color
from skimage.morphology import skeletonize, thin
from skimage import img_as_float

# Cek ketersediaan Google Colab
try:
    from google.colab import drive
    from google.colab.patches import cv2_imshow
    IN_COLAB = True
    print("Berjalan di Google Colab environment")
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print("Berjalan di Local environment (.venv)")
    def cv2_imshow(img, title=None):
        if len(img.shape) == 3:
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        else:
            plt.imshow(img, cmap="gray")
        if title:
            plt.title(title)
        plt.axis("off")
        plt.show()

# Penentuan direktori gambar adaptif (Colab Drive vs GitHub Repo Lokal)
def get_img_path(filename):
    candidates = [
        os.path.join("week_9/images", filename),
        os.path.join("images", filename),
        os.path.join("/content/drive/MyDrive/PCVK/Images", filename),
        os.path.join("/content/PCVK_2026/week_9/images", filename),
        os.path.join("week_7/images", filename),
        filename
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return f"/content/drive/MyDrive/PCVK/Images/{filename}"

print(f"Helper direktori citra aktif. Contoh path 'plat nomer.jpg': {get_img_path('plat nomer.jpg')}")

## C. ULASAN TEORI SINGKAT
1. **Morfologi Matematis**: Cabang ilmu analisis bentuk dan topologi struktur objek citra biner maupun grayscale melalui operasi spasial dengan **Structuring Element (SE)**.
2. **Erosi ($A \ominus B$)**: Operasi penyusutan/penipisan objek dengan mengikis piksel tepi; mempertahankan piksel hanya jika seluruh elemen SE bernilai 1 tepat bertumpuk pada foreground citra.
3. **Dilasi ($A \oplus B$)**: Operasi pembesaran/penebalan objek; piksel referensi bernilai 1 jika minimal ada satu elemen SE bernilai 1 yang bertumpuk pada foreground citra.
4. **Opening ($A \circ B = (A \ominus B) \oplus B$)**: Erosi diikuti dilasi; efektif melenyapkan bintik-bintik putih noise di latar belakang tanpa mengubah dimensi geometris utama objek.
5. **Closing ($A ullet B = (A \oplus B) \ominus B$)**: Dilasi diikuti erosi; efektif menutup lubang kecil (pinholes) dan menyambungkan retakan/goresan sempit pada objek.
6. **Top-Hat Transform ($f - (f \circ b)$)**: Mengisolasi elemen cerah yang lebih kecil daripada SE pada latar belakang dengan iluminasi gelap atau tidak merata.
7. **Black-Hat Transform ($(f ullet b) - f$)**: Mengisolasi elemen gelap yang lebih kecil daripada SE pada latar belakang cerah atau bervariasi.
8. **Skeletonization & Thinning**: Reduksi bentuk objek hingga setebal 1-2 piksel (rangka) sambil mempertahankan pusat posisi, konektivitas, dan topologi objek.

## D. TUGAS PRAKTIKUM (PERCOBAAN 1 SAMPAI 17)

### Percobaan 1 & 2: Setup Colab & Import Library
*Sudah diinisialisasi pada bagian cell persiapan di atas (Langkah 1 & 2).*

### Percobaan 3: Fungsi Dilasi Citra Tanpa Library (Custom Function - Structuring Element Cross 3x3)
**Instruksi:** Buat fungsi dilasi manual serta tampilkan hasil dilasi pada citra plat nomor dengan Structuring Element Dilasi (SED) berbentuk cross $3 \times 3$ tanpa menggunakan library morphology OpenCV.

In [ ]:
# Percobaan 3: Dilasi Citra Tanpa Library
def dilasi_citra(F, SED):
    # Dimensi citra input
    p, q = F.shape
    imgD = np.zeros((p, q), dtype=np.uint8)
    
    # Deteksi tipe Structuring Element (matriks biner atau ukuran skalar)
    if isinstance(SED, int):
        w = SED
        SED_matrix = np.ones((w, w), dtype=np.uint8)
    else:
        SED_matrix = np.array(SED, dtype=np.uint8)
        w = SED_matrix.shape[0]
        
    constant1 = (w - 1) // 2
    
    # Proses Dilasi: mengambil nilai maksimum piksel di dalam jendela SE aktif
    for i in range(constant1, p - constant1):
        for j in range(constant1, q - constant1):
            temp = F[i - constant1 : i + constant1 + 1, j - constant1 : j + constant1 + 1]
            product = temp * SED_matrix
            imgD[i, j] = np.max(product)
            
    return imgD

# Load citra plat nomor dalam format grayscale
img_d = cv2.imread(get_img_path('plat nomer.jpg'), 0)
ret, thresh = cv2.threshold(img_d, 125, 200, cv2.THRESH_BINARY)

# Structuring Element Cross 3x3
SED_cross3 = np.array([[0, 1, 0],
                       [1, 1, 1],
                       [0, 1, 0]], dtype=np.uint8)

w = 3
imgD = dilasi_citra(thresh, SED_cross3)

# Visualisasi perbandingan
plt.figure(figsize=(14, 4))
plt.subplot(131), plt.imshow(img_d, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(thresh, cmap='gray'), plt.title('Citra Biner'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(imgD, cmap='gray'), plt.title('Citra Hasil Dilasi (Cross 3x3 Manual)'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 4: Dilasi Citra Menggunakan Library Morphology OpenCV
**Instruksi:** Buat program implementasi operasi dilasi dengan Structuring Element Dilasi (SED) berbentuk cross $3 \times 3$ menggunakan fungsi `cv2.dilate` dari OpenCV.

In [ ]:
# Percobaan 4: Dilasi Menggunakan Library OpenCV
img = cv2.imread(get_img_path('plat nomer.jpg'))
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, thresh = cv2.threshold(img_gray, 127, 225, cv2.THRESH_BINARY)

# Structuring element Cross 3x3 via cv2.getStructuringElement
kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
dilasi = cv2.dilate(thresh, kernel, iterations=1)

print("Structuring Element (Cross 3x3 OpenCV):")
print(kernel)

plt.figure(figsize=(10, 4))
plt.subplot(121), plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), plt.title('Citra Awal')
plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(dilasi, cmap='gray'), plt.title('Citra Hasil Dilasi (OpenCV)')
plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 5: Fungsi Erosi Citra Tanpa Library (Custom Function - Structuring Element Square 5x5)
**Instruksi:** Buat fungsi erosi manual serta tampilkan hasil erosi pada citra plat nomor dengan Structuring Element Erosi (SE) berbentuk square $5 \times 5$ tanpa library OpenCV.

In [ ]:
# Percobaan 5: Erosi Tanpa Library (Manual)
def erosi_citra(F, k):
    m, n = F.shape
    if isinstance(k, int):
        SE = np.ones((k, k), dtype=np.uint8)
        w = k
    else:
        SE = np.array(k, dtype=np.uint8)
        w = SE.shape[0]
        
    constant = (w - 1) // 2
    imgE = np.zeros((m, n), dtype=np.uint8)
    
    # Proses Erosi: Foreground dipertahankan hanya jika seluruh elemen SE aktif bertumpuk pada foreground
    for i in range(constant, m - constant):
        for j in range(constant, n - constant):
            temp = F[i - constant : i + constant + 1, j - constant : j + constant + 1]
            if np.all(temp[SE == 1] > 0):
                imgE[i, j] = 255
            else:
                imgE[i, j] = 0
                
    return imgE

img_e = cv2.imread(get_img_path('plat nomer.jpg'), 0)
ret1, thresh1 = cv2.threshold(img_e, 175, 225, cv2.THRESH_BINARY)
k = 5
imgE = erosi_citra(thresh1, k)

plt.figure(figsize=(14, 4))
plt.subplot(131), plt.imshow(img_e, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(thresh1, cmap='gray'), plt.title('Citra Biner'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(imgE, cmap='gray'), plt.title('Citra Hasil Erosi (Square 5x5 Manual)'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 6: Erosi Citra Menggunakan Library Morphology OpenCV
**Instruksi:** Buat program implementasi operasi erosi dengan Structuring Element Square $5 \times 5$ menggunakan fungsi `cv2.erode` dari OpenCV.

In [ ]:
# Percobaan 6: Erosi Library OpenCV
img = cv2.imread(get_img_path('plat nomer.jpg'))
kernel = np.ones((5, 5), np.uint8)
erosion = cv2.erode(img, kernel, iterations=1)

print("Kernel Square 5x5 OpenCV:")
print(kernel)

plt.figure(figsize=(10, 4))
plt.subplot(121), plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), plt.title('Citra Awal')
plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(cv2.cvtColor(erosion, cv2.COLOR_BGR2RGB)), plt.title('Citra Hasil Erosi (OpenCV)')
plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 7: Operasi Opening dengan Structuring Element Square 3x3 (Tanpa vs Dengan Library OpenCV)
**Instruksi:** Buat program implementasi operasi opening ($A \circ B = (A \ominus B) \oplus B$) dengan SE Square $3 \times 3$ tanpa dan dengan library OpenCV.

In [ ]:
# Percobaan 7: Operasi Opening (Square 3x3)
img_o = cv2.imread(get_img_path('plat nomer.jpg'), 0)
reto, thresho = cv2.threshold(img_o, 127, 225, cv2.THRESH_BINARY)

# 1. Tanpa Library: Erosi -> Dilasi
k = 3
SE_sq3 = np.ones((k, k), dtype=np.uint8)
imgO_manual = dilasi_citra(erosi_citra(thresho, SE_sq3), SE_sq3)

# 2. Menggunakan Library OpenCV
kernel = np.ones((3, 3), np.uint8)
imgO_lib = cv2.morphologyEx(thresho, cv2.MORPH_OPEN, kernel)

# Visualisasi
plt.figure(figsize=(14, 4))
plt.subplot(141), plt.imshow(img_o, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(142), plt.imshow(thresho, cmap='gray'), plt.title('Citra Biner'), plt.xticks([]), plt.yticks([])
plt.subplot(143), plt.imshow(imgO_manual, cmap='gray'), plt.title('Opening (Manual)'), plt.xticks([]), plt.yticks([])
plt.subplot(144), plt.imshow(imgO_lib, cmap='gray'), plt.title('Opening (OpenCV)'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 8: Operasi Closing dengan Structuring Element Square 3x3 (Tanpa vs Dengan Library OpenCV)
**Instruksi:** Buat program implementasi operasi closing ($A \bullet B = (A \oplus B) \ominus B$) dengan SE Square $3 \times 3$ tanpa dan dengan library OpenCV.

In [ ]:
# Percobaan 8: Operasi Closing (Square 3x3)
img_c = cv2.imread(get_img_path('plat nomer.jpg'), 0)
retc, threshc = cv2.threshold(img_c, 127, 225, cv2.THRESH_BINARY)

# 1. Tanpa Library: Dilasi -> Erosi
k = 3
SE_sq3 = np.ones((k, k), dtype=np.uint8)
imgC_manual = erosi_citra(dilasi_citra(threshc, SE_sq3), SE_sq3)

# 2. Menggunakan Library OpenCV
kernel = np.ones((3, 3), np.uint8)
imgC_lib = cv2.morphologyEx(threshc, cv2.MORPH_CLOSE, kernel)

# Visualisasi
plt.figure(figsize=(14, 4))
plt.subplot(141), plt.imshow(img_c, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(142), plt.imshow(threshc, cmap='gray'), plt.title('Citra Biner'), plt.xticks([]), plt.yticks([])
plt.subplot(143), plt.imshow(imgC_manual, cmap='gray'), plt.title('Closing (Manual)'), plt.xticks([]), plt.yticks([])
plt.subplot(144), plt.imshow(imgC_lib, cmap='gray'), plt.title('Closing (OpenCV)'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 9: Operasi Dilasi dengan Structuring Element Cross 3x3 dan 5x5
**Instruksi:** Buat program implementasi operasi dilasi dengan Structuring Element Cross $3 \times 3$ dan $5 \times 5$.

In [ ]:
# Percobaan 9: Dilasi Cross 3x3 & Cross 5x5
img_d = cv2.imread(get_img_path('plat nomer.jpg'), 0)
ret, thresh = cv2.threshold(img_d, 127, 225, cv2.THRESH_BINARY)

# Matriks Cross 3x3
SED_cross3 = np.array([[0, 1, 0],
                       [1, 1, 1],
                       [0, 1, 0]], dtype=np.uint8)

# Matriks Cross 5x5
SED_cross5 = np.array([[0, 0, 1, 0, 0],
                       [0, 0, 1, 0, 0],
                       [1, 1, 1, 1, 1],
                       [0, 0, 1, 0, 0],
                       [0, 0, 1, 0, 0]], dtype=np.uint8)

imgD_c3 = dilasi_citra(thresh, SED_cross3)
imgD_c5 = dilasi_citra(thresh, SED_cross5)

plt.figure(figsize=(14, 4))
plt.subplot(131), plt.imshow(img_d, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(imgD_c3, cmap='gray'), plt.title('SED = Cross 3x3'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(imgD_c5, cmap='gray'), plt.title('SED = Cross 5x5'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 10: Operasi Dilasi dengan Structuring Element Circular (Disk/Ellipse) 3x3 dan 5x5
**Instruksi:** Buat program implementasi operasi dilasi dengan Structuring Element berbentuk Circular $3 \times 3$ dan $5 \times 5$.

In [ ]:
# Percobaan 10: Dilasi Circular 3x3 & 5x5
img_d = cv2.imread(get_img_path('plat nomer.jpg'), 0)
ret, thresh = cv2.threshold(img_d, 127, 225, cv2.THRESH_BINARY)

# Matriks Circular via cv2.MORPH_ELLIPSE
SED_circle3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
SED_circle5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

print("SED Circle 3x3:\n", SED_circle3)
print("SED Circle 5x5:\n", SED_circle5)

imgD_cir3 = dilasi_citra(thresh, SED_circle3)
imgD_cir5 = dilasi_citra(thresh, SED_circle5)

plt.figure(figsize=(14, 4))
plt.subplot(131), plt.imshow(img_d, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(imgD_cir3, cmap='gray'), plt.title('SED = Circle 3x3'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(imgD_cir5, cmap='gray'), plt.title('SED = Circle 5x5'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 11: Operasi Erosi dengan Structuring Element Rectangle 3x5 dan 5x7
**Instruksi:** Buat program implementasi operasi erosi dengan Structuring Element persegi panjang (*Rectangle*) ukuran $3 \times 5$ dan $5 \times 7$.

In [ ]:
# Percobaan 11: Erosi Rectangle 3x5 & 5x7
def erosi_citra_rect(F, p, l):
    m, n = F.shape
    SE = cv2.getStructuringElement(cv2.MORPH_RECT, (p, l))
    constant_y = (l - 1) // 2
    constant_x = (p - 1) // 2
    imgE = np.zeros((m, n), dtype=np.uint8)
    
    for i in range(constant_y, m - constant_y):
        for j in range(constant_x, n - constant_x):
            temp = F[i - constant_y : i + constant_y + 1, j - constant_x : j + constant_x + 1]
            if np.all(temp[SE == 1] > 0):
                imgE[i, j] = 255
            else:
                imgE[i, j] = 0
                
    return imgE

img_e = cv2.imread(get_img_path('plat nomer.jpg'), 0)
ret1, thresh1 = cv2.threshold(img_e, 127, 225, cv2.THRESH_BINARY)

# Rectangle 3x5 (lebar p=5, tinggi l=3)
imgE_rect35 = erosi_citra_rect(thresh1, p=5, l=3)

# Rectangle 5x7 (lebar p=7, tinggi l=5)
imgE_rect57 = erosi_citra_rect(thresh1, p=7, l=5)

plt.figure(figsize=(14, 4))
plt.subplot(131), plt.imshow(img_e, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(imgE_rect35, cmap='gray'), plt.title('SE = Rectangle 3x5'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(imgE_rect57, cmap='gray'), plt.title('SE = Rectangle 5x7'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 12: Operasi Erosi dengan Structuring Element Line Vertikal (Ukuran 3 dan 5)
**Instruksi:** Buat program implementasi operasi erosi dengan Structuring Element berbentuk Line Vertikal 3 dan 5.
- Efek: Elemen garis berarah vertikal dipertahankan, sedangkan elemen garis horizontal terkikis tajam.

In [ ]:
# Percobaan 12: Erosi Line Vertikal 3 & 5
def erosi_citra_line_v(F, k):
    m, n = F.shape
    SE = cv2.getStructuringElement(cv2.MORPH_RECT, (1, k)) # lebar 1, tinggi k
    constant = (k - 1) // 2
    imgE = np.zeros((m, n), dtype=np.uint8)
    
    for i in range(constant, m - constant):
        for j in range(n):
            temp = F[i - constant : i + constant + 1, j : j + 1]
            if np.all(temp[SE == 1] > 0):
                imgE[i, j] = 255
            else:
                imgE[i, j] = 0
                
    return imgE

img_e = cv2.imread(get_img_path('plat nomer.jpg'), 0)
ret1, thresh1 = cv2.threshold(img_e, 127, 225, cv2.THRESH_BINARY)

imgE_v3 = erosi_citra_line_v(thresh1, k=3)
imgE_v5 = erosi_citra_line_v(thresh1, k=5)

plt.figure(figsize=(14, 4))
plt.subplot(131), plt.imshow(img_e, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(imgE_v3, cmap='gray'), plt.title('SE = Line Vertikal 3'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(imgE_v5, cmap='gray'), plt.title('SE = Line Vertikal 5'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 13: Transformasi Top-Hat (White Hat)
**Instruksi:** Menerapkan transformasi Top-Hat pada citra.
- Rumus: $TopHat(f) = f - (f \circ b)$
- Kegunaan: Mengekstrak elemen-elemen cerah yang ukurannya lebih kecil dari structuring element pada latar belakang yang tidak merata.
*(Catatan: Variabel `input_image` dari modul praktikum diperbaiki menjadi variabel citra `img_cat_gray`).*

In [ ]:
# Percobaan 13: Top-Hat Transform
filterSize = (3, 3)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, filterSize)

# Menggunakan citra kitten01.jpg
img_cat = cv2.imread(get_img_path('kitten01.jpg'))
if img_cat is None:
    img_cat = cv2.imread(get_img_path('plat nomer.jpg'))
img_cat_gray = cv2.cvtColor(img_cat, cv2.COLOR_BGR2GRAY) if len(img_cat.shape) == 3 else img_cat

# Menerapkan operasi Top-Hat
tophat_img = cv2.morphologyEx(img_cat_gray, cv2.MORPH_TOPHAT, kernel)

plt.figure(figsize=(10, 4))
plt.subplot(121), plt.imshow(img_cat_gray, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(tophat_img, cmap='gray'), plt.title('Citra Hasil TOP HAT'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 14: Transformasi Black-Hat (Bottom-Hat)
**Instruksi:** Menerapkan transformasi Black-Hat pada citra.
- Rumus: $BlackHat(f) = (f \bullet b) - f$
- Kegunaan: Mengekstrak elemen-elemen gelap yang lebih kecil dari structuring element pada latar belakang cerah.

In [ ]:
# Percobaan 14: Black-Hat Transform
filterSize = (3, 3)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, filterSize)

img_cat = cv2.imread(get_img_path('kitten01.jpg'))
if img_cat is None:
    img_cat = cv2.imread(get_img_path('plat nomer.jpg'))
img_cat_gray = cv2.cvtColor(img_cat, cv2.COLOR_BGR2GRAY) if len(img_cat.shape) == 3 else img_cat

# Kernel berukuran (13, 5) sesuai modul praktikum
Kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
blackhat_img = cv2.morphologyEx(img_cat_gray, cv2.MORPH_BLACKHAT, Kernel)

plt.figure(figsize=(10, 4))
plt.subplot(121), plt.imshow(img_cat_gray, cmap='gray'), plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(blackhat_img, cmap='gray'), plt.title('Citra Hasil Black Hat'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 15: Operasi Skeletonization
**Instruksi:** Ekstraksi rangka (*skeleton*) dari objek citra menggunakan `skimage.morphology.skeletonize` pada citra bunga lily (`lily.jpg`).

In [ ]:
# Percobaan 15: Skeletonization
img_lily = cv2.imread(get_img_path('lily.jpg'), 0)

# Binarisasi dengan Triangle thresholding
binary = img_lily > filters.threshold_triangle(img_lily)

# Salin citra biner & jadikan tipe integer 0 dan 1
binary_cp = binary.copy().astype(np.uint8)

# Skeletonize citra
skeleton = skeletonize(binary_cp)

plt.figure(figsize=(12, 5))
plt.subplot(121), plt.imshow(img_lily, cmap='gray'), plt.title('Original Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(skeleton, cmap='gray'), plt.title('Skeleton Image'), plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

### Percobaan 16: Skeleton Inverse
**Instruksi:** Ekstraksi rangka dengan bagian gelap (*black bits*) sebagai foreground (`img < threshold`) dan tampilkan dengan colormap terbalik (`gray_r`).

In [ ]:
# Percobaan 16: Skeleton Inverse
img_lily_io = io.imread(get_img_path('lily.jpg'))
if len(img_lily_io.shape) == 3:
    img_gray_io = img_lily_io[..., 0]
else:
    img_gray_io = img_lily_io

binary_inv = img_gray_io < filters.threshold_triangle(img_gray_io)
skeleton_inv = skeletonize(binary_inv)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_gray_io, cmap='gray')
ax[0].set_title('Original Image')
ax[0].set_axis_off()

ax[1].imshow(skeleton_inv, cmap='gray_r')
ax[1].set_title('Skeleton (Inverse)')
ax[1].set_axis_off()
plt.tight_layout()
plt.show()

### Percobaan 17: Thickening & Thinning
**Instruksi:** Implementasikan operasi penipisan (*thinning*) dan pengekstrakan kerangka pada citra biner menggunakan fungsi `skimage.morphology.thin`.

In [ ]:
# Percobaan 17: Thinning & Skeletonize
img_raw = io.imread(get_img_path('lily.jpg'))
if len(img_raw.shape) == 3:
    image_float = img_as_float(color.rgb2gray(img_raw))
else:
    image_float = img_as_float(img_raw)

image_binary = image_float < 0.5
out_skeletonize = morphology.skeletonize(image_binary)
out_thin = morphology.thin(image_binary)

f, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4))
ax0.imshow(image_float, cmap='gray')
ax0.set_title('Input')
ax0.axis('off')

ax1.imshow(out_thin, cmap='gray')
ax1.set_title('Thinning Result')
ax1.axis('off')
plt.tight_layout()
plt.show()

## E. TUGAS PRAKTIKUM: IMPLEMENTASI MORFOLOGI PADA DATASET KTP
**Deskripsi Tugas Modul:**
*Tugas Praktikum Morfologi melibatkan implementasi dataset KTP yang terdapat dalam Modul 7. Dalam tugas ini, kita akan menggunakan semua informasi yang disediakan dalam modul Morfologi dan menerapkan sejumlah nilai tambahan dalam kode sumber. Tujuannya adalah agar citra KTP dapat terbaca dengan baik ketika teknik morfologi diterapkan pada citra tersebut.*

### Analisis Permasalahan Citra KTP:
1. **Latar Belakang Gradien & Watermark:** Latar belakang KTP memiliki gradasi warna biru serta pola watermark burung garuda dan ornamen peta yang mengganggu binarisasi.
2. **Kelemahan Thresholding Murni (Modul 7):** Penggunaan Otsu atau Adaptive thresholding secara langsung menghasilkan banyak noise bercak (bocor) serta membuat beberapa goresan teks terputus (*broken characters*).
3. **Solusi Pendekatan Morfologi:**
   - **Black-Hat Transform ($21 \times 21$):** Mengisolasi karakter teks gelap dan meratakan variasi pencahayaan latar belakang.
   - **Binarisasi Otsu:** Mengubah citra hasil Black-Hat menjadi biner secara akurat.
   - **Opening ($2 \times 2$):** Menghilangkan titik-titik noise putih kecil di area latar belakang.
   - **Closing ($2 \times 2$):** Menyambung celah goresan huruf yang terputus.
   - **Dilation / Thickening ($2 \times 1$):** Mempertegas dan menebalkan kontur teks agar solid dan jelas terbaca.

In [ ]:
# E. Implementasi Lengkap Pipeline Morfologi Citra KTP
# 1. Baca Citra KTP
img_ktp = cv2.imread(get_img_path('ktp_fulan.jpg'))
ktp_gray = cv2.cvtColor(img_ktp, cv2.COLOR_BGR2GRAY)

# 2. Koreksi Iluminasi & Ekstraksi Fitur Teks Gelap menggunakan Black-Hat Transform
kernel_bg = cv2.getStructuringElement(cv2.MORPH_RECT, (19, 19))
ktp_blackhat = cv2.morphologyEx(ktp_gray, cv2.MORPH_BLACKHAT, kernel_bg)

# 3. Binarisasi Optimal (Otsu Thresholding) pada Hasil Black-Hat
_, ktp_bin = cv2.threshold(ktp_blackhat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 4. Morfologi Opening: Membersihkan noise bintik putih kecil di background
se_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
ktp_opening = cv2.morphologyEx(ktp_bin, cv2.MORPH_OPEN, se_open)

# 5. Morfologi Closing: Menutup celah kecil dan menyambungkan goresan huruf yang terputus
se_close = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
ktp_closing = cv2.morphologyEx(ktp_opening, cv2.MORPH_CLOSE, se_close)

# 6. Dilasi Terkendali (Stroke Thickening): Menebalkan goresan huruf agar solid dan tegas
se_thick = np.ones((2, 1), dtype=np.uint8) # penguatan vertikal ringan
ktp_enhanced = cv2.dilate(ktp_closing, se_thick, iterations=1)

# Inversi untuk format dokumen standar (teks hitam di atas latar putih)
ktp_doc = cv2.bitwise_not(ktp_enhanced)

# Baseline Modul 7 (Otsu murni tanpa morfologi) sebagai pembanding
_, ktp_otsu_modul7 = cv2.threshold(ktp_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Visualisasi Perbandingan Menyeluruh
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes[0, 0].imshow(cv2.cvtColor(img_ktp, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('(a) Citra Asli KTP')
axes[0, 0].axis('off')

axes[0, 1].imshow(ktp_otsu_modul7, cmap='gray')
axes[0, 1].set_title('(b) Baseline Modul 7 (Otsu Tanpa Morfologi - Noisy & Bocor)')
axes[0, 1].axis('off')

axes[1, 0].imshow(ktp_blackhat, cmap='gray')
axes[1, 0].set_title('(c) Tahap 1: Black-Hat Transform (Iluminasi Latar Dihilangkan)')
axes[1, 0].axis('off')

axes[1, 1].imshow(ktp_opening, cmap='gray')
axes[1, 1].set_title('(d) Tahap 2: Binarisasi + Opening (Noise Bintik Latar Dihapus)')
axes[1, 1].axis('off')

axes[2, 0].imshow(ktp_enhanced, cmap='gray')
axes[2, 0].set_title('(e) Tahap 3: Closing + Dilasi (Karakter Tersambung & Tebal)')
axes[2, 0].axis('off')

axes[2, 1].imshow(ktp_doc, cmap='gray')
axes[2, 1].set_title('(f) Hasil Akhir Dokumen (Invers: Teks Hitam di Atas Putih - Sangat Jelas)')
axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

### Pembahasan & Jawaban Pertanyaan Analisis Morfologi KTP (Singkat & To-The-Point)

1. **Mengapa operasi Black-Hat Transform sangat efektif untuk citra KTP?**
   - **Jawaban:** Karakter teks pada KTP berwarna gelap di atas latar belakang cerah bergradasi ($f_{teks} < f_{bg}$). Black-Hat menghitung $(f \bullet b) - f$, yang mengeliminasi variasi iluminasi dan pola watermark skala besar, sehingga hanya menyisakan komponen teks berukuran kecil dengan kontras tinggi.

2. **Apa fungsi kombinasi Opening dan Closing pada pipeline ini?**
   - **Opening ($2 \times 2$):** Mengikis dan melenyapkan bintik-bintik putih (*salt noise*) pada latar belakang tanpa merusak struktur karakter.
   - **Closing ($2 \times 2$):** Menjembatani putusnya goresan huruf (*broken strokes*) akibat kontras rendah dan mengisi lubang mikro di dalam tubuh karakter.

3. **Mengapa hasil morfologi jauh lebih unggul dibandingkan metode Thresholding murni (Modul 7)?**
   - **Jawaban:** Thresholding murni (Modul 7) hanya membedakan intensitas piksel skalar secara independen tanpa memahami hubungan spasial/bentuk geometri objek, sehingga pola watermark dan gradasi foto KTP ikut terbinarisasi menjadi noise kotor. Morfologi memproses bentuk dan skala struktural objek (*spatial structuring element*), memisahkan teks dari noise latar belakang secara bersih dan presisi.